In [1]:
# If needed, install deps (uncomment as needed).
# !pip install ultralytics opencv-python tqdm pandas pyyaml

import sys, platform, torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
PyTorch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 4090


In [2]:
from pathlib import Path

# 🔁 Change this root if you want the project somewhere else
ROOT = Path("/home/akif/ComputerVision/Assignment_8/widerface_yolo")

DATA_RAW = ROOT / "data_raw"          # where ZIPs and/or extracted folders live
DATASET  = ROOT / "dataset"           # YOLO-formatted dataset that we'll create
IM_TRAIN_DST = DATASET / "images" / "train"
IM_VAL_DST   = DATASET / "images" / "val"
LB_TRAIN_DST = DATASET / "labels" / "train"
LB_VAL_DST   = DATASET / "labels" / "val"

for p in [DATA_RAW, IM_TRAIN_DST, IM_VAL_DST, LB_TRAIN_DST, LB_VAL_DST]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DATA_RAW:", DATA_RAW)
print("DATASET:", DATASET)

ROOT: /home/akif/ComputerVision/Assignment_8/widerface_yolo
DATA_RAW: /home/akif/ComputerVision/Assignment_8/widerface_yolo/data_raw
DATASET: /home/akif/ComputerVision/Assignment_8/widerface_yolo/dataset


In [3]:
# 👇 Adjust these ONLY if your extracted folders are elsewhere
WIDER_TRAIN_IMAGES = DATA_RAW / "WIDER_train" / "images"
WIDER_VAL_IMAGES   = DATA_RAW / "WIDER_val"   / "images"
SPLIT_DIR          = DATA_RAW / "wider_face_split"

TRAIN_SPLIT_TXT = SPLIT_DIR / "wider_face_train_bbx_gt.txt"
VAL_SPLIT_TXT   = SPLIT_DIR / "wider_face_val_bbx_gt.txt"

print("Train images dir exists:", WIDER_TRAIN_IMAGES.exists())
print("Val   images dir exists:", WIDER_VAL_IMAGES.exists())
print("Split dir exists:", SPLIT_DIR.exists())
print("Train txt exists:", TRAIN_SPLIT_TXT.exists())
print("Val   txt exists:", VAL_SPLIT_TXT.exists())

Train images dir exists: True
Val   images dir exists: True
Split dir exists: True
Train txt exists: True
Val   txt exists: True


In [4]:
# OPTIONAL: Try downloading from known mirrors (requires internet).
# If this fails, skip and do it from terminal, then re-run Cell 2.

import requests, zipfile, io, os

def fetch_zip(url: str, dst_dir: Path, expect_dir_name: str = None):
    dst_dir.mkdir(parents=True, exist_ok=True)
    print("Downloading:", url)
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    zf.extractall(dst_dir)
    print("Unzipped into:", dst_dir)

# Example mirrors (may change). Comment/uncomment as needed.
# fetch_zip("https://huggingface.co/datasets/ashudeep/WIDER_FACE/resolve/main/WIDER_train.zip", DATA_RAW)
# fetch_zip("https://huggingface.co/datasets/ashudeep/WIDER_FACE/resolve/main/WIDER_val.zip",   DATA_RAW)
# fetch_zip("https://huggingface.co/datasets/ashudeep/WIDER_FACE/resolve/main/wider_face_split.zip", DATA_RAW)

In [5]:
import cv2, shutil
from tqdm import tqdm

MIN_W, MIN_H = 8, 8   # drop ultra-small boxes (<8 px) to reduce label noise

def parse_wider_split(split_file, img_src_root, img_dst_root, lbl_dst_root):
    """
    WIDER split file format:
      <relative/image/path.jpg>
      <num_faces>
      x y w h blur expression illumination invalid occlusion pose
      x y w h ...
      --
    Repeats for each image.
    We use only (x,y,w,h). One class: face -> id 0.
    """
    with open(split_file, "r") as f:
        lines = f.read().strip().splitlines()

    i, n = 0, len(lines)
    num_images = dropped_boxes = kept_boxes = 0

    # heuristic: count images for progress bar
    total_blocks = sum(1 for line in lines if line.strip().endswith(".jpg"))
    pbar = tqdm(total=total_blocks, desc=f"Converting {split_file.name}", leave=False)

    while i < n:
        rel_path = lines[i].strip(); i += 1
        if not rel_path.endswith(".jpg"):
            continue

        # number of faces on next line
        if i >= n:
            break
        try:
            num_faces = int(lines[i].strip()); i += 1
        except:
            # malformed; skip to next probable block
            continue

        src_img = img_src_root / rel_path
        if not src_img.exists():
            # consume bbox lines & skip
            i += num_faces
            continue

        dst_img = img_dst_root / rel_path
        dst_img.parent.mkdir(parents=True, exist_ok=True)
        if not dst_img.exists():
            shutil.copy2(src_img, dst_img)

        im = cv2.imread(str(dst_img))
        if im is None:
            i += num_faces
            continue
        H, W = im.shape[:2]

        dst_lbl = (lbl_dst_root / rel_path).with_suffix(".txt")
        dst_lbl.parent.mkdir(parents=True, exist_ok=True)
        with open(dst_lbl, "w") as lf:
            for _ in range(num_faces):
                if i >= n: break
                parts = lines[i].strip().split(); i += 1
                if len(parts) < 4:
                    continue
                x, y, w, h = map(float, parts[:4])
                # skip invalid/tiny
                if w <= 0 or h <= 0 or w < MIN_W or h < MIN_H:
                    dropped_boxes += 1
                    continue

                # YOLO normalized (cx, cy, w, h)
                cx = (x + w/2) / W
                cy = (y + h/2) / H
                nw = w / W
                nh = h / H

                # clamp
                cx = min(max(cx, 0), 1)
                cy = min(max(cy, 0), 1)
                nw = min(max(nw, 0), 1)
                nh = min(max(nh, 0), 1)

                lf.write(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")
                kept_boxes += 1

        num_images += 1
        pbar.update(1)

    pbar.close()
    return dict(images=num_images, boxes_kept=kept_boxes, boxes_dropped=dropped_boxes)

# Run conversion
train_stats = parse_wider_split(TRAIN_SPLIT_TXT, WIDER_TRAIN_IMAGES, IM_TRAIN_DST, LB_TRAIN_DST)
val_stats   = parse_wider_split(VAL_SPLIT_TXT,   WIDER_VAL_IMAGES,   IM_VAL_DST,   LB_VAL_DST)

print("Train stats:", train_stats)
print("Val   stats:", val_stats)

Train stats: {'images': 12880, 'boxes_kept': 132589, 'boxes_dropped': 26831}
Val   stats: {'images': 3226, 'boxes_kept': 32674, 'boxes_dropped': 7034}


In [6]:
import yaml

yaml_path = ROOT / "widerface.yaml"
data_cfg = {
    "path": str(DATASET),
    "train": "images/train",
    "val":   "images/val",
    "names": {0: "face"},
    "nc": 1
}
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("Created:", yaml_path)
print(yaml_path.read_text())

Created: /home/akif/ComputerVision/Assignment_8/widerface_yolo/widerface.yaml
path: /home/akif/ComputerVision/Assignment_8/widerface_yolo/dataset
train: images/train
val: images/val
names:
  0: face
nc: 1



In [7]:
from ultralytics import YOLO

MODEL  = "yolov8n.pt"     # try: 'yolov8s.pt' or 'yolov8m.pt'
IMGSZ  = 800              # 640 default; 800/1024 can help small faces
EPOCHS = 75
BATCH  = 16               # tune for your GPU
PROJECT= str(ROOT / "runs")
NAME   = "yolov8n_widerface"

model = YOLO(MODEL)
train_res = model.train(
    data=str(yaml_path),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=PROJECT,
    name=NAME,
    pretrained=True,
    cos_lr=True,
    lr0=0.01,
    patience=20,  # early stop if no val improvement
    # device=0,   # uncomment to force GPU:0; leave to auto
)
train_res

Ultralytics 8.3.213 🚀 Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 22684MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/akif/ComputerVision/Assignment_8/widerface_yolo/widerface.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=75, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_widerface4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, o

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b1a154ef6a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [8]:
best_weights = ROOT / "runs" / NAME / "weights" / "best.pt"
assert best_weights.exists(), f"Missing: {best_weights}"

model = YOLO(str(best_weights))
val_res = model.val(
    data=str(yaml_path),
    imgsz=IMGSZ,
    batch=BATCH,
    plots=True,
    project=PROJECT,
    name=f"{NAME}_val"
)
val_res  # prints metrics; plots saved under runs/{NAME}_val

Ultralytics 8.3.213 🚀 Python-3.10.18 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 22684MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5121.8±1899.5 MB/s, size: 166.2 KB)
val: Scanning /home/akif/ComputerVision/Assignment_8/widerface_yolo/dataset/labels/val/0--Parade.cache... 3226 images, 6 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3226/3226 12.8Mit/s 0.0ss
val: /home/akif/ComputerVision/Assignment_8/widerface_yolo/dataset/images/val/21--Festival/21_Festival_Festival_21_604.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 202/202 26.1it/s 7.7s<0.0s
                   all       3226      32673      0.794      0.599      0.678      0.352
Speed: 0.1ms preprocess, 0.8ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/runs/yolov8n_widerf

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b1a1a964220>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [9]:
import pandas as pd
from pathlib import Path

# Paths
train_csv = ROOT / "runs" / NAME / "results.csv"
val_dir   = ROOT / "runs" / f"{NAME}_val"

# Safely find a results.csv inside val_dir
val_csv = None
if val_dir.exists():
    matches = list(val_dir.glob("results.csv"))
    if matches:
        val_csv = matches[0]

dfs = []
if train_csv.exists():
    df_train = pd.read_csv(train_csv)
    dfs.append(("train_last_epoch", df_train.tail(1)))
if val_csv is not None and val_csv.exists():
    df_val = pd.read_csv(val_csv)
    dfs.append(("val", df_val.tail(1)))

def pick(df, cols):
    return df[[c for c in cols if c in df.columns]]

cols = ["precision", "recall", "map50", "map75", "map", "epoch", "box_loss", "cls_loss"]
for name, df in dfs:
    print(f"\n== {name} ==")
    display(pick(df, cols))

# If dfs is empty, no CSVs were found
if not dfs:
    print("⚠ No results.csv found in train or val folders. Did training/validation complete?")


== train_last_epoch ==


,epoch
1,2


In [10]:
from ultralytics import YOLO
from tqdm import tqdm

pred_out = ROOT / "pred_samples"
pred_out.mkdir(parents=True, exist_ok=True)

model = YOLO(str(best_weights))
sample_imgs = list(IM_VAL_DST.rglob("*.jpg"))[:12]  # first 12 samples
for p in tqdm(sample_imgs, desc="Predicting"):
    model.predict(
        source=str(p),
        conf=0.25,
        imgsz=IMGSZ,
        save=True,
        save_txt=False,
        project=str(pred_out),
        name="",
        exist_ok=True,
        verbose=False
    )

print("Saved annotated predictions to:", pred_out)

Predicting:   0%|          | 0/12 [00:00<?, ?it/s]

Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict
Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict
Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict


Predicting:  25%|██▌       | 3/12 [00:00<00:00, 26.95it/s]

Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict
Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict
Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict
Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict
Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict


Predicting:  67%|██████▋   | 8/12 [00:00<00:00, 39.62it/s]

Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict
Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict
Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict
Results saved to /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples/predict


Predicting: 100%|██████████| 12/12 [00:00<00:00, 42.06it/s]

Saved annotated predictions to: /home/akif/ComputerVision/Assignment_8/widerface_yolo/pred_samples
